# Scaled-up, night/dark-robust joint training (road + building + point)

**Cloud GPU only -- do not run locally.** This notebook is the moderate-scale, multi-epoch
successor to `train_unet_joint.ipynb` (which is a frozen 2-epoch/~150-tile CPU proof of concept).
Its goal is specifically **robustness to dark and night satellite imagery**, which the proof-of-
concept model was never exposed to and fails on.

Two independent lines of defense against dark/night input, combined (per the project's night-
robustness plan):

1. **Real illumination-invariant data**: SpaceNet 6 (`datasets/spacenet6.py`) -- Capella Space SAR
   imagery over Rotterdam. SAR is an active sensor (it illuminates its own target with radar), so
   it is genuinely illumination-invariant -- it looks the same day or night. Building-footprint
   labels only (SN6 has no road/point annotations), folded in as a third `SOURCE_CLASSES` entry
   with zero architecture change (converted to pseudo-RGB at dataset-build time, see
   `datasets/spacenet6.py::sar_bands_to_pseudo_rgb`).
2. **Synthetic low-light augmentation** (`datasets/augment.py::simulate_low_light`) applied to a
   fraction of training patches cut from the daytime-labeled SpaceNet/Potsdam sources -- gamma
   darkening, brightness/contrast/saturation jitter, CLAHE, and sensor-noise injection, since no
   public dataset exists with real night-labeled optical satellite imagery paired with full
   road+building+point semantic labels (verified during planning).

A dedicated **dark-test evaluation** section near the end applies the same low-light simulation at
fixed severities to the real held-out test tiles and reports `evaluate_all()` metrics side by side
across severities -- this is the concrete evidence for whether the model actually got more robust,
not just that dark examples were included in training.

**Scale** (moderate, per project scope decision): ~1,000-1,500 base tiles across 3 sources
(SpaceNet SN2/SN3 x4 cities, Potsdam, SpaceNet6), patchified into many thousands of 256x256
patches, ~40 epochs, mixed precision, on a single CUDA GPU -- should run in hours, not require
multi-GPU. Never modifies the frozen `train_unet.ipynb`/`train_unet_joint.ipynb` baselines; new
checkpoints land at `models/unet_joint_4class_scaled*.pt`.


## Cloud GPU setup (mandatory -- this notebook has no CPU fallback)

Unlike the frozen CPU baselines, this notebook is GPU-only by design: a full run at this scale on
CPU would take far too long to be practical. The cell below clones the repo if it isn't already
present, installs all dependencies (including a CUDA-enabled `torch` build -- the rest of the repo
documents a CPU-wheel install, since the baselines are CPU-only; this notebook explicitly needs the
GPU wheel instead), and hard-fails if no CUDA device is visible rather than silently falling back to
CPU and taking hours/days longer than expected.


In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/DanielZGeorge/PLEM.git"
REPO_DIR = "/content/PLEM" if "google.colab" in sys.modules else os.path.abspath("..")

code_present = (os.path.isdir(os.path.join(REPO_DIR, "datasets"))
                and os.path.isdir(os.path.join(REPO_DIR, "models")))

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
elif code_present:
    # Uploaded directly (e.g. a zipped copy of the repo pushed to a cloud GPU box) rather
    # than git-cloned -- the code is already here, so skip cloning/pulling entirely instead
    # of failing on a non-empty destination.
    print(f"Using manually uploaded copy at {REPO_DIR} (no .git dir found, skipping clone/pull).")
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                os.path.join(REPO_DIR, "requirements.txt")], check=True)

# CUDA-enabled torch build -- replaces the CPU-wheel instruction the rest of
# the repo's CLAUDE.md documents, since this notebook is GPU-only by design.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tqdm"], check=True)

os.chdir(os.path.join(REPO_DIR, "notebooks"))
print(f"Repo ready at {REPO_DIR}, cwd set to {os.getcwd()}")


In [ ]:
import sys, os, random
from pathlib import Path
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader

assert torch.cuda.is_available(), (
    "No CUDA device visible -- this notebook is GPU-only by design (unlike the frozen CPU "
    "baselines). Run it on a cloud GPU server, not locally."
)

from metrics import evaluate_all
from models.unet import SmallUNet
from losses.multitask import PLEMMultiTaskLoss
from datasets.joint import load_joint_tiles, class_mask_for_source, SOURCE_CLASSES, NUM_CLASSES
from datasets.spacenet import build_spacenet_sample
from datasets.potsdam import load_kaggle_dataset, build_potsdam_sample
from datasets.spacenet6 import build_spacenet6_sample
from datasets.augment import simulate_low_light

DATA_DIR = Path("..").resolve() / "data"
MODELS_DIR = Path("..").resolve() / "models"
MODELS_DIR.mkdir(exist_ok=True)

SEED = 0
PATCH = 256
DEVICE = torch.device("cuda")
print(f"Using device: {DEVICE} ({torch.cuda.get_device_name(0)})")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

CLASS_NAMES = {0: "background", 1: "road", 2: "building", 3: "point"}


def overlay4(label):
    """Color-code a 4-class label map: road red, building green, point yellow."""
    o = np.zeros((*label.shape, 3), dtype=np.uint8)
    o[label == 1] = [255, 0, 0]
    o[label == 2] = [0, 255, 0]
    o[label == 3] = [255, 255, 0]
    return o


## Build the SpaceNet + Potsdam + SpaceNet6 caches (moderate scale)

Unconditional (not Colab-gated) -- a fresh cloud box always starts with an empty `data/` cache, so
this always needs to run once. Skips a source entirely if its cache directory already has files
(so re-running this cell after an interrupted run is cheap). Scale targets, within the project's
approved "moderate" band:

- **SpaceNet** (`datasets/spacenet.py`): 4 cities x `n_tiles=175` (up from the proof-of-concept's
  25/city) -- road+building, no point class.
- **Potsdam** (`datasets/potsdam.py`): `n_crops=200` is a non-binding cap -- the Kaggle mirror's
  actual available crop count is the real limiter (the full Potsdam sample is much smaller than
  200; this just avoids artificially capping below what's actually available). Building+point, no
  road class. Needs a one-time, user-side Kaggle API token (`~/.kaggle/kaggle.json`) -- if missing,
  this cell prints instructions and continues with SpaceNet+SpaceNet6-only data rather than
  crashing.
- **SpaceNet6** (`datasets/spacenet6.py`, new): `n_tiles=300` -- real SAR imagery, building only,
  the illumination-invariant half of the night-robustness approach. Downloads one large tarball
  (multi-GB, one-time) rather than many small per-tile objects.


In [ ]:
CITIES = ["Vegas", "Khartoum", "Paris", "Shanghai"]


def _count_source_npz(name):
    """Cached .npz count using the SAME globbing load_joint_tiles uses, so this
    matches what actually loads. A recursive rglob would over-count a cache
    written to the wrong depth and hide a layout bug."""
    d = DATA_DIR / name
    if not d.is_dir():
        return 0
    if name == "spacenet":
        n = sum(1 for cd in d.iterdir() if cd.is_dir() for _ in cd.glob("*.npz"))
        return n or len(list(d.rglob("*.npz")))  # rglob fallback mirrors load_joint_tiles
    return len(list(d.glob("*.npz")))


if not os.path.isdir(DATA_DIR / "spacenet") or not any((DATA_DIR / "spacenet").iterdir()):
    print("data/spacenet is empty -- building the scaled SpaceNet sample (SN2+SN3)...")
    for city in CITIES:
        # cache_dir WITHOUT the city: build_spacenet_sample appends <city>/ itself
        # (datasets/spacenet.py), and load_joint_tiles expects data/spacenet/<city>/*.npz.
        # Passing DATA_DIR/"spacenet"/city here produced data/spacenet/<city>/<city>/*.npz
        # and load_joint_tiles found zero SpaceNet tiles.
        build_spacenet_sample(city, n_tiles=175, n_road_candidates=400,
                               cache_dir=str(DATA_DIR / "spacenet"), seed=0)
else:
    print("data/spacenet already populated -- skipping.")

# Fail fast BEFORE the multi-GB SpaceNet6 download if SpaceNet yielded nothing.
assert _count_source_npz("spacenet") > 0, (
    "0 SpaceNet tiles cached -- the build failed or the cache landed at an "
    "unexpected path. Fix this before continuing; the SN6 download below is expensive."
)

if not os.path.isdir(DATA_DIR / "potsdam") or not any((DATA_DIR / "potsdam").iterdir()):
    try:
        load_kaggle_dataset(dest=str(DATA_DIR / "potsdam_raw"))
        build_potsdam_sample(
            n_crops=200, seed=0, cache_dir=str(DATA_DIR / "potsdam"),
            raw_dir=str(DATA_DIR / "potsdam_raw"), extract_buildings=True,
        )
    except Exception as e:
        print(f"Could not build the Potsdam cache automatically ({e}). Set up "
              f"~/.kaggle/kaggle.json (see potsdam_data_prep.ipynb) and re-run this cell.")
else:
    print("data/potsdam already populated -- skipping.")

# Fail fast here too -- cell 7's `assert len(by_source) == 3` requires Potsdam, and
# catching a zero-yield now (rather than after the 42 GB SN6 pull) saves an hour.
assert _count_source_npz("potsdam") > 0, (
    "0 Potsdam tiles cached -- the point class can't be trained/evaluated and the "
    "3-source assert later will fail. Set up ~/.kaggle/kaggle.json and re-run this cell."
)

if not os.path.isdir(DATA_DIR / "spacenet6") or not any((DATA_DIR / "spacenet6").iterdir()):
    print("data/spacenet6 is empty -- building the SpaceNet6 SAR sample (this downloads a "
          "multi-GB tarball once)...")
    build_spacenet6_sample(
        n_tiles=300, cache_dir=str(DATA_DIR / "spacenet6"), seed=0,
        # explicit tarball_path -- the function's own default is a relative string
        # that would resolve against notebooks/ (cell 2 chdir'd there). Anchor it
        # to the top-level data/ dir like every other cache path here.
        tarball_path=str(DATA_DIR / "spacenet6_raw" / "SN6_buildings_AOI_11_Rotterdam_train.tar.gz"),
    )
else:
    print("data/spacenet6 already populated -- skipping.")

for name in ("spacenet", "potsdam", "spacenet6"):
    print(f"  {name}: {_count_source_npz(name)} cached tiles")


## Load tiles and split train/val/test (per source, then concatenate)

`load_joint_tiles()` now loads all **three** sources, tagging each with its `source` and
`class_mask` (`datasets/joint.py::SOURCE_CLASSES` -- `spacenet6` added as a third entry,
building-only). Each source is split 70/15/15 **independently** at the tile level before
concatenating, same rationale as the proof-of-concept notebook: a single pooled-then-permuted split
risks an unlucky seed starving the smallest source from a split entirely.


In [ ]:
def split_tiles(tiles, seed=SEED):
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(tiles))
    n = len(tiles)
    n_train = int(round(0.70 * n))
    n_val = int(round(0.15 * n))
    train = [tiles[i] for i in perm[:n_train]]
    val = [tiles[i] for i in perm[n_train:n_train + n_val]]
    test = [tiles[i] for i in perm[n_train + n_val:]]
    return train, val, test


all_tiles = load_joint_tiles(
    spacenet_dir=DATA_DIR / "spacenet", potsdam_dir=DATA_DIR / "potsdam",
    spacenet6_dir=DATA_DIR / "spacenet6",
)
by_source = {}
for t in all_tiles:
    by_source.setdefault(t["source"], []).append(t)
print(f"Loaded {len(all_tiles)} tiles: " + ", ".join(f"{k}={len(v)}" for k, v in by_source.items()))

train_tiles, val_tiles, test_tiles = [], [], []
for source, tiles in by_source.items():
    tr, va, te = split_tiles(tiles, seed=SEED)
    train_tiles += tr
    val_tiles += va
    test_tiles += te
    print(f"  {source}: train={len(tr)} val={len(va)} test={len(te)}")

print(f"total: train={len(train_tiles)}  val={len(val_tiles)}  test={len(test_tiles)}")
assert len(by_source) == 3, (
    f"Expected all 3 sources (spacenet, potsdam, spacenet6) to be present, got {list(by_source)} "
    f"-- check the dataset-build cell above for a source that failed silently."
)


## Patchify

Identical to `train_unet_joint.ipynb`'s `pad_to_multiple`/`patchify_tile`/`predict_tile` -- these
are already source-agnostic (a patch just carries its source tile's fixed `class_mask` through
unchanged), so a third source needs no changes here.


In [ ]:
def pad_to_multiple(image, label, patch=PATCH):
    h, w = label.shape
    new_h = int(np.ceil(h / patch)) * patch
    new_w = int(np.ceil(w / patch)) * patch
    pad_h, pad_w = new_h - h, new_w - w
    image_p = np.pad(image, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")
    label_p = np.pad(label, ((0, pad_h), (0, pad_w)), mode="constant", constant_values=0)
    return image_p, label_p, (h, w)


def patchify_tile(image, label, class_mask, patch=PATCH):
    """Returns a list of (image_patch, label_patch, class_mask) covering the whole tile."""
    image_p, label_p, _ = pad_to_multiple(image, label, patch)
    ph, pw = label_p.shape
    patches = []
    for r in range(0, ph, patch):
        for c in range(0, pw, patch):
            patches.append((
                image_p[r:r + patch, c:c + patch], label_p[r:r + patch, c:c + patch], class_mask,
            ))
    return patches


def predict_tile(model, image, orig_label_shape, patch=PATCH, device=DEVICE):
    """Runs the model patch-by-patch over a full tile and stitches an HxW prediction map."""
    dummy_label = np.zeros(orig_label_shape, dtype=np.uint8)
    image_p, _, orig_shape = pad_to_multiple(image, dummy_label, patch)
    ph, pw = image_p.shape[:2]
    canvas = np.zeros((ph, pw), dtype=np.uint8)
    model.eval()
    with torch.no_grad():
        for r in range(0, ph, patch):
            for c in range(0, pw, patch):
                patch_img = image_p[r:r + patch, c:c + patch]
                x = torch.from_numpy(patch_img.transpose(2, 0, 1).astype(np.float32) / 255.0)
                x = x.unsqueeze(0).to(device)
                logits = model(x)
                pred = logits.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
                canvas[r:r + patch, c:c + patch] = pred
    h, w = orig_shape
    return canvas[:h, :w]


train_patches = [p for t in train_tiles for p in patchify_tile(t["image"], t["label"], t["class_mask"])]
val_patches = [p for t in val_tiles for p in patchify_tile(t["image"], t["label"], t["class_mask"])]
print(f"train_patches={len(train_patches)}  val_patches={len(val_patches)}  (patch={PATCH}px)")


## Dataset / DataLoader with dark-augmentation

`PatchDataset` gains a `dark_aug_prob` parameter: on each `__getitem__` call (train split only),
with probability `dark_aug_prob` the image is darkened via `simulate_low_light` (randomized
severity, drawn fresh per call) *before* the existing flip augmentation -- the label is never
touched, since severity is purely a pixel-value transform. A per-batch source-mix sanity check
(distinguishing all 3 sources by their distinct `class_mask` patterns:
spacenet=`[1,1,1,0]`, potsdam=`[1,0,1,1]`, spacenet6=`[1,0,1,0]`) confirms all 3 sources actually
appear across training batches.


In [ ]:
class PatchDataset(Dataset):
    def __init__(self, patches, augment=False, dark_aug_prob=0.0):
        self.patches = patches
        self.augment = augment
        self.dark_aug_prob = dark_aug_prob
        self._rng = np.random.default_rng(SEED)

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        image, label, class_mask = self.patches[idx]
        if self.dark_aug_prob > 0 and random.random() < self.dark_aug_prob:
            image = simulate_low_light(image, self._rng)
        if self.augment:
            if random.random() < 0.5:
                image, label = image[:, ::-1], label[:, ::-1]
            if random.random() < 0.5:
                image, label = image[::-1, :], label[::-1, :]
        image_t = torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1)).astype(np.float32) / 255.0)
        label_t = torch.from_numpy(np.ascontiguousarray(label).astype(np.int64))
        class_mask_t = torch.from_numpy(np.ascontiguousarray(class_mask).astype(np.float32))
        return image_t, label_t, class_mask_t


BATCH_SIZE = 48
DARK_AUG_PROB = 0.4

train_loader = DataLoader(
    PatchDataset(train_patches, augment=True, dark_aug_prob=DARK_AUG_PROB),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    PatchDataset(val_patches, augment=False, dark_aug_prob=0.0),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True,
)

# Per-batch source-mix sanity check: each source has a distinct class_mask
# pattern (see markdown above), so class_mask alone identifies the source.
# islice, not list(...)[:3] -- the latter materializes a full training epoch
# (every batch + a full pass of dark augmentation) just to print 3 rows.
import itertools
first_batches = list(itertools.islice(train_loader, 3))
for i, (_, _, cm) in enumerate(first_batches):
    n_spacenet = int(((cm[:, 1] == 1) & (cm[:, 3] == 0)).sum())
    n_sn6 = int(((cm[:, 1] == 0) & (cm[:, 3] == 0)).sum())
    n_potsdam = int((cm[:, 1] == 0).sum()) - n_sn6
    print(f"batch {i}: spacenet={n_spacenet}  potsdam={n_potsdam}  spacenet6={n_sn6}  "
          f"(batch_size={cm.shape[0]})")


## 4-class U-Net + `PLEMMultiTaskLoss` (unchanged)

Same `SmallUNet` backbone and `PLEMMultiTaskLoss` configuration as `train_unet_joint.ipynb` -- the
per-source class-masking mechanism already generalizes to a third source with zero code change
here (see `datasets/joint.py::SOURCE_CLASSES`).


In [ ]:
model = SmallUNet(in_ch=3, num_classes=NUM_CLASSES, base=16).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"SmallUNet (4-class) parameter count: {n_params:,}")

CLASS_CONFIG = {
    1: {"name": "road", "tolerance": 10},
    2: {"name": "building", "tolerance": 2},
    3: {"name": "point", "tolerance": 3},
}
loss_fn = PLEMMultiTaskLoss(
    class_config=CLASS_CONFIG, linear_classes=[1], polygon_classes=[2], point_classes=[3],
)


## Training loop -- GPU-scaled, multi-epoch, mixed precision, periodic checkpointing, early stopping

Unlike the 2-epoch CPU proof of concept, this is a real multi-epoch GPU run long enough to be worth
interrupting/resuming, so it gets: automatic mixed precision (`torch.cuda.amp`), a cosine LR
schedule, a periodic rolling checkpoint every `CHECKPOINT_EVERY` epochs (in addition to the
existing best-val-loss checkpoint) so an interrupted run doesn't lose all progress, and **early
stopping** on validation loss (`PATIENCE = 3` epochs, deliberately tight/sensitive rather than the
more common 5-10 -- this run's `EPOCHS=40` budget is a ceiling, not a target, and stopping promptly
once val_loss stops improving is preferred over burning cloud-GPU hours on a plateaued run). Note
the `CosineAnnealingLR` schedule is still built for the full `T_max=EPOCHS`, so an early-stopped run
ends before the LR has fully annealed to its floor -- an accepted trade-off, not a bug.


In [ ]:
EPOCHS = 40
LR = 2e-3
CHECKPOINT_EVERY = 5
PATIENCE = 3  # sensitive: stop after 3 epochs with no val_loss improvement
MIN_DELTA = 1e-4  # improvement smaller than this doesn't reset the patience counter
BEST_CHECKPOINT_PATH = MODELS_DIR / "unet_joint_4class_scaled.pt"
LATEST_CHECKPOINT_PATH = MODELS_DIR / "unet_joint_4class_scaled_latest.pt"

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()

history = {"epoch": [], "train_loss": [], "val_loss": []}
sub_term_history = {"epoch": []}
best_val_loss = float("inf")
epochs_no_improve = 0
stopped_early = False

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum, train_n = 0.0, 0
    sub_sums = {}
    for images, labels, class_masks in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        class_masks = class_masks.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            out = loss_fn(model(images), labels, class_masks)
        scaler.scale(out["loss"]).backward()
        scaler.step(optimizer)
        scaler.update()
        bs = images.size(0)
        train_loss_sum += out["loss"].item() * bs
        train_n += bs
        for k, v in out.items():
            if k == "loss":
                continue
            sub_sums[k] = sub_sums.get(k, 0.0) + v * bs
    scheduler.step()
    train_loss = train_loss_sum / train_n

    model.eval()
    val_loss_sum, val_n = 0.0, 0
    with torch.no_grad():
        for images, labels, class_masks in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            class_masks = class_masks.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast():
                out = loss_fn(model(images), labels, class_masks)
            val_loss_sum += out["loss"].item() * images.size(0)
            val_n += images.size(0)
    val_loss = val_loss_sum / val_n

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    sub_term_history["epoch"].append(epoch)
    for k, total in sub_sums.items():
        sub_term_history.setdefault(k, []).append(total / train_n)

    sub_str = "  ".join(f"{k}={total / train_n:.4f}" for k, total in sub_sums.items())
    print(f"epoch {epoch:3d}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"lr={scheduler.get_last_lr()[0]:.2e}  [{sub_str}]")

    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), BEST_CHECKPOINT_PATH)
        print(f"  -> new best val_loss, checkpoint saved to {BEST_CHECKPOINT_PATH}")
    else:
        epochs_no_improve += 1
        print(f"  -> no improvement for {epochs_no_improve}/{PATIENCE} epoch(s) "
              f"(best_val_loss={best_val_loss:.4f})")

    if epoch % CHECKPOINT_EVERY == 0 or epoch == EPOCHS:
        torch.save({
            "epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
        }, LATEST_CHECKPOINT_PATH)
        print(f"  -> rolling checkpoint saved to {LATEST_CHECKPOINT_PATH} (resumable)")

    if epochs_no_improve >= PATIENCE:
        stopped_early = True
        print(f"\nEarly stopping: val_loss hasn't improved by >= {MIN_DELTA} for "
              f"{PATIENCE} consecutive epochs (stopped after epoch {epoch}/{EPOCHS}).")
        break

model.load_state_dict(torch.load(BEST_CHECKPOINT_PATH, map_location=DEVICE))
model.eval()
status = "stopped early" if stopped_early else "ran to EPOCHS"
print(f"\nBest val_loss={best_val_loss:.4f} ({status}); loaded that checkpoint for evaluation.")

sub_term_df = pd.DataFrame(sub_term_history)
print("\nPer-epoch sub-term train loss (verification: are ALL terms decreasing, not just the total?):")
display(sub_term_df)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history["epoch"], history["train_loss"], color="#2a78d6", label="train_loss")
ax.plot(history["epoch"], history["val_loss"], color="#eb6834", label="val_loss")
ax.set_xlabel("epoch")
ax.set_ylabel("PLEMMultiTaskLoss (total)")
ax.set_title("Joint 4-class scaled training curve")
ax.legend()
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for k in ["ce_dice", "tolerance", "cldice", "heatmap"]:
    if k in sub_term_df.columns:
        ax.plot(sub_term_df["epoch"], sub_term_df[k], label=k)
ax.set_xlabel("epoch")
ax.set_ylabel("train loss (sub-term)")
ax.set_title("Per-term training loss -- each should trend down, not just the total")
ax.legend()
fig.tight_layout()
plt.show()


## Test-set evaluation with PLEM's metrics (normal brightness)

Same pattern as `train_unet_joint.ipynb`, now with a 3-valued `source` column.


In [ ]:
import hashlib


def stable_seed(s):
    """Process-stable 32-bit seed from a string. Python's built-in hash() is
    salted per process, so `abs(hash(x)) % 2**32` is not reproducible run to run
    despite the "fixed severities" claim in the dark-test section."""
    return int(hashlib.sha256(str(s).encode()).hexdigest()[:8], 16)


# Score each tile ONLY on the feature types its own source annotates. Passing the
# point class in a shared dtaf1_config (or linear_classes=[1] to a road-less
# source) drags DTAF1/CBHM to ~0 for Potsdam/SpaceNet6 regardless of prediction
# quality -- see metrics/unified.py::evaluate_all on why the point class is
# deliberately kept out of the shared config. cbhm() now degenerates to the
# building score when linear_classes is empty instead of collapsing to 0.
_SOURCE_EVAL = {
    "spacenet":  dict(linear_classes=[1], polygon_classes=[2], point_classes=None,
                      dtaf1_config={k: CLASS_CONFIG[k] for k in (1, 2)}),
    "potsdam":   dict(linear_classes=[], polygon_classes=[2], point_classes=[3],
                      dtaf1_config={k: CLASS_CONFIG[k] for k in (2, 3)}),
    "spacenet6": dict(linear_classes=[], polygon_classes=[2], point_classes=None,
                      dtaf1_config={k: CLASS_CONFIG[k] for k in (2,)}),
}

_METRIC_COLS = ["cbhm", "cbhm_soft", "dtaf1", "dtaf1_weighted",
                "cldice_mean", "bf_mean", "point_f1_mean"]


def evaluate_tile(pred, gt, source):
    # point_min_area=2 drops single-pixel speckle in the point channel (a dark/SAR
    # prediction can scatter 10^4+ 1-px blobs and stall the dark-test sweep).
    return evaluate_all(pred, gt, point_min_area=2, **_SOURCE_EVAL[source])


test_rows = []
test_preds = {}

for t in test_tiles:
    pred = predict_tile(model, t["image"], t["label"].shape)
    test_preds[t["tile"]] = pred
    result = evaluate_tile(pred, t["label"], t["source"])
    test_rows.append({
        "source": t["source"], "tile": t["tile"],
        **{k: result[k] for k in _METRIC_COLS},
    })

test_results = pd.DataFrame(test_rows)
for col in _METRIC_COLS:
    test_results[col] = pd.to_numeric(test_results[col], errors="coerce")
test_results.to_csv(DATA_DIR / "train_unet_joint_scaled_test_results.csv", index=False)

# For Potsdam/SpaceNet6, cbhm reduces to the building score; point_f1_mean is only
# meaningful for Potsdam (NaN elsewhere). Report the informative columns per source,
# not just cbhm/dtaf1.
print(test_results.groupby("source")[
    ["cbhm", "cbhm_soft", "dtaf1", "bf_mean", "point_f1_mean"]
].mean())
test_results


In [ ]:

# Verification: training is per-source masked (PLEMMultiTaskLoss only supervises the
# classes each tile's source actually annotates -- see SOURCE_CLASSES/class_mask above),
# but prediction is NOT source-conditioned: predict_tile() always runs a plain unmasked
# argmax over all 4 channels regardless of t["source"]. (evaluate_tile() above IS scored
# per-source -- only on the classes each source annotates -- but that's a metric-reporting
# choice, not something predict_tile sees.) This block makes the inference guarantee
# explicit rather than implicit in the code path.
import inspect

sig = inspect.signature(predict_tile)
assert "class_mask" not in sig.parameters and "source" not in sig.parameters, (
    "predict_tile() must not accept a class_mask/source argument -- predictions are required "
    "to span all 3 non-background dimensions (road/building/point) on every test tile "
    "regardless of which classes that tile's own source provides ground truth for. If this "
    "assert fires, someone added source-conditioned masking to inference and broken that "
    "guarantee."
)

class_pixel_counts = {}  # source -> {class_id: predicted pixel count}
for t in test_tiles:
    counts = class_pixel_counts.setdefault(t["source"], {1: 0, 2: 0, 3: 0})
    pred = test_preds[t["tile"]]
    for c in (1, 2, 3):
        counts[c] += int((pred == c).sum())

pred_pixel_counts = pd.DataFrame([
    {"source": source, **{CLASS_NAMES[c]: n for c, n in counts.items()}}
    for source, counts in sorted(class_pixel_counts.items())
])
print("Predicted pixel counts per class, per test-tile source:")
display(pred_pixel_counts)

print("\nClasses each source's own ground truth never annotates, and whether the model "
      "predicted them anyway (evidence of full-dimension, GT-independent prediction):")
for source in sorted(by_source):
    annotated = set(SOURCE_CLASSES[source])
    unannotated = {1, 2, 3} - annotated
    if not unannotated:
        print(f"  {source}: annotates all 3 classes -- nothing to check.")
        continue
    row = pred_pixel_counts.loc[pred_pixel_counts["source"] == source].iloc[0]
    for c in sorted(unannotated):
        px = int(row[CLASS_NAMES[c]])
        print(f"  {source}: predicted {CLASS_NAMES[c]!r} on {px} px "
              f"(a class this source's ground truth never contains).")


## Dark-test evaluation: does the model actually get more robust to dark/night input?

This is the headline evidence for the night-robustness claim, not just "dark examples were
included in training." Each real test tile's image is darkened via `simulate_low_light` at fixed,
reproducible severities (`0.0` = untouched baseline), then run through the same stitched-inference
+ `evaluate_all()` pipeline as the normal-brightness evaluation above. A model that's actually
learned to be robust should degrade gracefully as severity increases, not collapse the way an
untrained-for-darkness model would.


In [ ]:
DARK_TEST_SEVERITIES = [0.0, 0.3, 0.6, 0.9]
DARK_PLOT_METRICS = ["cbhm", "dtaf1", "bf_mean", "point_f1_mean"]
_METRIC_COLORS = {"cbhm": "#2a78d6", "dtaf1": "#eb6834",
                  "bf_mean": "#2ca25f", "point_f1_mean": "#8856a7"}
dark_rows = []

for severity in DARK_TEST_SEVERITIES:
    for t in test_tiles:
        if severity == 0.0:
            img = t["image"]
        else:
            img = simulate_low_light(
                t["image"], np.random.default_rng(stable_seed(t["tile"])), severity=severity,
            )
        pred = predict_tile(model, img, t["label"].shape)
        result = evaluate_tile(pred, t["label"], t["source"])
        dark_rows.append({
            "source": t["source"], "tile": t["tile"], "severity": severity,
            "cbhm": result["cbhm"], "dtaf1": result["dtaf1"],
            "cldice_mean": result["cldice_mean"], "bf_mean": result["bf_mean"],
            "point_f1_mean": result["point_f1_mean"],
        })

dark_results = pd.DataFrame(dark_rows)
for col in ["cbhm", "dtaf1", "cldice_mean", "bf_mean", "point_f1_mean"]:
    dark_results[col] = pd.to_numeric(dark_results[col], errors="coerce")
dark_results.to_csv(DATA_DIR / "train_unet_joint_scaled_dark_test_results.csv", index=False)

dark_summary = dark_results.groupby(["source", "severity"])[DARK_PLOT_METRICS].agg(["mean", "std"])
print("Mean +/- std, per (source, severity) -- higher severity = darker input.")
print("cbhm reduces to the building score for potsdam/spacenet6; point_f1_mean is potsdam-only.")
display(dark_summary)

fig, axes = plt.subplots(1, len(by_source), figsize=(5.5 * len(by_source), 4), sharey=True)
if len(by_source) == 1:
    axes = [axes]
for ax, source in zip(axes, sorted(by_source)):
    sub = dark_results[dark_results["source"] == source]
    means = sub.groupby("severity")[DARK_PLOT_METRICS].mean()
    stds = sub.groupby("severity")[DARK_PLOT_METRICS].std()
    for metric in DARK_PLOT_METRICS:
        if means[metric].notna().any():
            ax.plot(means.index, means[metric], marker="o",
                    color=_METRIC_COLORS[metric], label=metric)
            ax.fill_between(means.index, means[metric] - stds[metric].fillna(0),
                             means[metric] + stds[metric].fillna(0),
                             color=_METRIC_COLORS[metric], alpha=0.15)
    ax.set_title(source)
    ax.set_xlabel("dark-simulation severity")
    ax.legend()
axes[0].set_ylabel("metric value")
fig.suptitle("Metric vs. darkening severity, per source (mean +/- std band)")
fig.tight_layout()
plt.show()


## Qualitative review: image / GT / prediction, including one darkened example

Same 3-panel-per-tile layout as `train_unet_joint.ipynb`, with one additional row showing a
darkened version of a test tile alongside its prediction -- a direct visual check of the dark-test
table above.


In [ ]:
N_EXAMPLES = min(3, len(test_tiles))
example_order = test_results.sort_values("cbhm")["tile"].tolist()
example_stems = [example_order[i] for i in np.linspace(0, len(example_order) - 1, N_EXAMPLES).round().astype(int)]
tiles_by_stem = {t["tile"]: t for t in test_tiles}

n_rows = N_EXAMPLES + 1  # +1 for the darkened example row
fig, axes = plt.subplots(n_rows, 3, figsize=(13.5, 4.5 * n_rows))
if n_rows == 1:
    axes = axes[None, :]

for row, stem in enumerate(example_stems):
    t = tiles_by_stem[stem]
    gt, pred, image = t["label"], test_preds[stem], t["image"]
    scores = test_results.loc[test_results["tile"] == stem].iloc[0]

    axes[row, 0].imshow(image)
    axes[row, 0].imshow(overlay4(gt), alpha=0.5)
    axes[row, 0].set_title(f"{stem} ({t['source']}) -- GT")
    axes[row, 1].imshow(image)
    axes[row, 1].imshow(overlay4(pred), alpha=0.5)
    axes[row, 1].set_title(f"{stem} -- prediction")
    axes[row, 2].imshow(overlay4(pred))
    axes[row, 2].set_title("raw prediction mask")
    for col in range(3):
        axes[row, col].axis("off")

    pf1 = scores["point_f1_mean"]
    pf1_str = f"{pf1:.3f}" if pd.notna(pf1) else "n/a"
    print(f"{stem} ({t['source']}):  cbhm={scores['cbhm']:.3f}  dtaf1={scores['dtaf1']:.3f}  "
          f"point_f1_mean={pf1_str}")

# Darkened example row (severity=0.6, matching the dark-test sweep).
dark_row = n_rows - 1
dark_stem = example_stems[0]
t = tiles_by_stem[dark_stem]
dark_image = simulate_low_light(
    t["image"], np.random.default_rng(stable_seed(dark_stem)), severity=0.6,
)
dark_pred = predict_tile(model, dark_image, t["label"].shape)

axes[dark_row, 0].imshow(dark_image)
axes[dark_row, 0].imshow(overlay4(t["label"]), alpha=0.5)
axes[dark_row, 0].set_title(f"{dark_stem} (darkened, severity=0.6) -- GT")
axes[dark_row, 1].imshow(dark_image)
axes[dark_row, 1].imshow(overlay4(dark_pred), alpha=0.5)
axes[dark_row, 1].set_title(f"{dark_stem} (darkened) -- prediction")
axes[dark_row, 2].imshow(overlay4(dark_pred))
axes[dark_row, 2].set_title("raw prediction mask (darkened input)")
for col in range(3):
    axes[dark_row, col].axis("off")

fig.tight_layout()
plt.show()


## Observations

- First joint 0D/1D/2D training run over **three** heterogeneous sources (SpaceNet: road+building;
  Potsdam: building+point; SpaceNet6: building only, real SAR) -- `SOURCE_CLASSES`'s per-source
  masking mechanism generalized to a third source with zero changes to `losses/multitask.py` or
  `models/unet.py`, confirming the design decision documented in `CLAUDE.md`.
- **Night/dark-robustness evidence lives in the dark-test section above**, not in the normal-
  brightness test results -- that's the comparison that actually speaks to whether this run
  improved on the failure mode that motivated it (the proof-of-concept model failing on dark/night
  imagery).
- **Early stopping** (`PATIENCE = 3`, `MIN_DELTA = 1e-4`, deliberately tight) means `EPOCHS=40` is a
  ceiling, not a guaranteed epoch count -- check the training-loop cell's printed "stopped early" vs.
  "ran to EPOCHS" status before assuming any run trained the full budget. Because `CosineAnnealingLR`
  is still built for `T_max=EPOCHS`, an early-stopped run's LR schedule never reaches its floor --
  worth keeping in mind if a `PATIENCE=3` stop looks premature; a larger `PATIENCE` or a schedule
  keyed to actual stopped-epoch count is the fix if that turns out to matter in practice.
- **Known limitation, inherited and now spanning 3 sources**: SpaceNet, Potsdam, and SpaceNet6 are
  not resampled to a common GSD, and SpaceNet6's SAR imagery is additionally a different sensor
  modality entirely, compressed to pseudo-RGB via `sar_bands_to_pseudo_rgb`'s `"replicate"` mode --
  which discards SAR's phase/polarimetric structure that a dedicated encoder branch could have
  exploited. Accepted trade-off given this project's moderate scope; a real ablation of this choice
  (e.g. a small dedicated SAR encoder branch) is future work, not done here.
- **Comparison against `train_unet_joint.ipynb`'s 2-epoch/2-source run is directional only**:
  different epoch budget, different dataset scale/composition, and different random seed dynamics
  -- not an apples-to-apples ablation. The strongest follow-up evidence that the augmentation
  specifically (not just more data/epochs) helped would be a `DARK_AUG_PROB=0` re-run compared
  against this one on the same dark-test sweep -- noted as future work, not run here.
- If a `PLEMMultiTaskLoss` sub-term's per-epoch value stays flat near its starting value while the
  total still drops, that term isn't learning -- worth checking its `weights` entry before trusting
  its contribution to the final model (same caveat as the proof-of-concept notebook).
